# Rank-three kernel and conditional queries

This notebook first evaluates joint FSNM estimation when the three true singular values are close. It then reuses the learned kernel to answer several conditional queries without refitting the model.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import ParameterGrid
from fsnm import empirical_loss, fit_fsnm


def basis_matrix(values, rank=3):
    values = np.asarray(values)
    basis = np.column_stack(
        [
            np.sqrt(2) * np.sin(np.pi * values),
            np.sqrt(2) * np.cos(np.pi * values),
            np.sqrt(2) * np.sin(2 * np.pi * values),
        ]
    )
    return basis[:, :rank]


def kappa_exact(x_values, y_values):
    return 1 + (basis_matrix(x_values) * SIGMAS) @ basis_matrix(y_values).T


def sample_joint(size, seed):
    rng = np.random.default_rng(seed)
    upper_bound = 1 + 2 * SIGMAS.sum()
    x_parts = []
    y_parts = []
    n_accepted = 0

    while n_accepted < size:
        x = rng.uniform(-1, 1, size)
        y = rng.uniform(-1, 1, size)
        density_ratio = 1 + np.sum(
            basis_matrix(x) * SIGMAS * basis_matrix(y), axis=1
        )
        accepted = rng.uniform(size=size) < density_ratio / upper_bound
        x_parts.append(x[accepted])
        y_parts.append(y[accepted])
        n_accepted += accepted.sum()

    return np.concatenate(x_parts)[:size], np.concatenate(y_parts)[:size]


def scaled_factors(phi_model, psi_model, singular_values, x, y):
    scale = np.sqrt(np.maximum(singular_values, 0))
    return phi_model.predict(x[:, None]) * scale, psi_model.predict(y[:, None]) * scale


def subspace_error(estimated, exact):
    estimated = estimated - estimated.mean(axis=0)
    exact = exact - exact.mean(axis=0)
    estimated_basis = np.linalg.qr(estimated)[0][:, : exact.shape[1]]
    exact_basis = np.linalg.qr(exact)[0][:, : exact.shape[1]]
    difference = estimated_basis @ estimated_basis.T - exact_basis @ exact_basis.T
    return np.linalg.norm(difference, ord="fro") / np.sqrt(2 * exact.shape[1])


def orthogonality_error(values):
    centered = values - values.mean(axis=0)
    gram = centered.T @ centered / len(centered)
    return np.linalg.norm(gram - np.eye(gram.shape[0]), ord="fro") / np.sqrt(gram.shape[0])

We use

$$\kappa(x,y)=1+\sum_{j=1}^3\sigma_j e_j(x)e_j(y),$$

where $e_1(t)=\sqrt{2}\sin(\pi t)$, $e_2(t)=\sqrt{2}\cos(\pi t)$, and $e_3(t)=\sqrt{2}\sin(2\pi t)$. We use $(\sigma_1,\sigma_2,\sigma_3)=(0.18,0.16,0.12)$. Since $|e_j(x)e_j(y)|\leq 2$, the density ratio is at least $1-2\sum_j\sigma_j=0.08$.

In [2]:
SIGMAS = np.array([0.18, 0.16, 0.12])
RANK = len(SIGMAS)
N_TRAIN = 5_000
N_VALIDATION = 2_000
N_REFERENCE = 5_000
SEED = 0

x_train, y_train = sample_joint(N_TRAIN, seed=12)
x_validation, y_validation = sample_joint(N_VALIDATION, seed=99)
_, y_reference = sample_joint(N_REFERENCE, seed=123)

In [3]:
TUNE_HYPERPARAMETERS = False

DEFAULT_FSNM_PARAMETERS = {
    "n_iterations": 10,
    "step_size": 0.1,
    "max_depth": 3,
    "min_samples_leaf": 300,
}
FSNM_PARAMETER_GRID = {
    "n_iterations": [10, 20, 40],
    "step_size": [0.05, 0.1, 0.2],
    "max_depth": [3, None],
    "min_samples_leaf": [100, 300],
}

if TUNE_HYPERPARAMETERS:
    tuning_results = []
    for candidate in ParameterGrid(FSNM_PARAMETER_GRID):
        phi_candidate, psi_candidate, values_candidate, _ = fit_fsnm(
            x_train, y_train, rank=RANK, seed=SEED, **candidate
        )
        phi_valid, psi_valid = scaled_factors(
            phi_candidate, psi_candidate, values_candidate, x_validation, y_validation
        )
        tuning_results.append((float(empirical_loss(phi_valid, psi_valid)), candidate))
    best_validation_loss, fsnm_parameters = min(tuning_results, key=lambda result: result[0])
    print(f"Best tuning loss: {best_validation_loss:.4f}")
else:
    fsnm_parameters = DEFAULT_FSNM_PARAMETERS.copy()

print(f"Hyperparameter tuning: {TUNE_HYPERPARAMETERS}")
print(f"FSNM hyperparameters: {fsnm_parameters}")

Hyperparameter tuning: False
FSNM hyperparameters: {'n_iterations': 10, 'step_size': 0.1, 'max_depth': 3, 'min_samples_leaf': 300}


In [4]:
phi_fsnm, psi_fsnm, values_fsnm, history_fsnm = fit_fsnm(
    x_train, y_train, rank=RANK, seed=SEED, **fsnm_parameters
)
phi_validation, psi_validation = scaled_factors(
    phi_fsnm, psi_fsnm, values_fsnm, x_validation, y_validation
)
validation_loss = float(empirical_loss(phi_validation, psi_validation))

grid = np.linspace(-1, 1, 160)
exact_basis = basis_matrix(grid)
kappa_true = kappa_exact(grid, grid)
phi_fsnm_grid = phi_fsnm.predict(grid[:, None])
psi_fsnm_grid = psi_fsnm.predict(grid[:, None])
kappa_fsnm = 1 + (phi_fsnm_grid * values_fsnm) @ psi_fsnm_grid.T

metrics = {
    "RMSE": np.sqrt(np.mean((kappa_fsnm - kappa_true) ** 2)),
    "spectrum error": np.linalg.norm(values_fsnm - SIGMAS),
    "subspace error": 0.5 * (
        subspace_error(phi_fsnm_grid, exact_basis)
        + subspace_error(psi_fsnm_grid, exact_basis)
    ),
    "orthogonality error": 0.5 * (
        orthogonality_error(phi_fsnm_grid)
        + orthogonality_error(psi_fsnm_grid)
    ),
}

print(f"True singular values: {SIGMAS}")
print(f"FSNM singular values: {np.round(values_fsnm, 4)}")
print(f"FSNM validation loss: {validation_loss:.4f}")
print("\nMetric                 FSNM")
for metric, value in metrics.items():
    print(f"{metric:20s} {value:9.4f}")

True singular values: [0.18 0.16 0.12]
FSNM singular values: [0.1818 0.1353 0.1018]
FSNM validation loss: -0.0465

Metric                 FSNM
RMSE                    0.1322
spectrum error          0.0308
subspace error          0.3724
orthogonality error     0.0254


In [5]:
value_limits = (
    min(kappa_true.min(), kappa_fsnm.min()),
    max(kappa_true.max(), kappa_fsnm.max()),
)
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), constrained_layout=True)
for axis, (values, title) in zip(
    axes[:2],
    [
        (kappa_true, r"True $\kappa$"),
        (kappa_fsnm, r"FSNM $\widehat\kappa$"),
    ],
):
    image = axis.imshow(
        values.T,
        origin="lower",
        extent=[-1, 1, -1, 1],
        cmap="coolwarm",
        vmin=value_limits[0],
        vmax=value_limits[1],
    )
    axis.set(title=title, xlabel="$x$", ylabel="$y$")
    fig.colorbar(image, ax=axis, shrink=0.82)

error = kappa_fsnm - kappa_true
error_limit = np.max(np.abs(error))
image = axes[2].imshow(
    error.T,
    origin="lower",
    extent=[-1, 1, -1, 1],
    cmap="coolwarm",
    vmin=-error_limit,
    vmax=error_limit,
)
axes[2].set(title=r"$\widehat\kappa-\kappa$", xlabel="$x$", ylabel="$y$")
fig.colorbar(image, ax=axes[2], shrink=0.82)

project_directory = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
figure_directory = project_directory / "figures"
figure_directory.mkdir(exist_ok=True)
figure_path = figure_directory / "00_rank3_close_spectrum.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {figure_path}")

iterations = np.arange(1, len(history_fsnm["training_loss"]) + 1)
loss_figure, loss_axis = plt.subplots(figsize=(5.5, 3.6), constrained_layout=True)
loss_axis.plot(iterations, history_fsnm["training_loss"], color="tab:blue", linewidth=2)
loss_axis.set(xlabel="Iteration", ylabel="Empirical training loss")
loss_axis.grid(alpha=0.25)
loss_path = figure_directory / "00_rank3_training_loss.png"
loss_figure.savefig(loss_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {loss_path}")

integration_grid = np.linspace(-1, 1, 4_001)
kappa_true_queries = kappa_exact(grid, integration_grid)
psi_reference = psi_fsnm.predict(y_reference[:, None])
kappa_fsnm_queries = 1 + (phi_fsnm_grid * values_fsnm) @ psi_reference.T

def exact_conditional(query_values):
    return 0.5 * np.trapezoid(
        kappa_true_queries * query_values[None, :],
        integration_grid,
        axis=1,
    )


def estimated_conditional(query_values):
    return np.mean(kappa_fsnm_queries * query_values[None, :], axis=1)


exact_mean = exact_conditional(integration_grid)
estimated_mean = estimated_conditional(y_reference)
exact_second_moment = exact_conditional(integration_grid**2)
estimated_second_moment = estimated_conditional(y_reference**2)
exact_variance = exact_second_moment - exact_mean**2
estimated_variance = estimated_second_moment - estimated_mean**2
exact_tail = exact_conditional((integration_grid > 0.5).astype(float))
estimated_tail = estimated_conditional((y_reference > 0.5).astype(float))

query_results = [
    (r"$\mathbb{E}[Y\mid X=x]$", exact_mean, estimated_mean),
    (r"$\mathrm{Var}(Y\mid X=x)$", exact_variance, estimated_variance),
    (r"$\mathbb{P}(Y>0.5\mid X=x)$", exact_tail, estimated_tail),
]
query_errors = {
    title: np.sqrt(np.mean((estimated - exact) ** 2))
    for title, exact, estimated in query_results
}

query_figure, query_axes = plt.subplots(
    1, 3, figsize=(12, 3.2), constrained_layout=True
)
for axis, (title, exact, estimated) in zip(query_axes, query_results):
    axis.plot(grid, exact, color="black", linewidth=2, label="Exact")
    axis.plot(grid, estimated, color="tab:red", linewidth=2, linestyle="--", label="FSNM")
    axis.set(title=title, xlabel="$x$")
query_axes[0].set_ylabel("Conditional functional")
query_axes[0].legend(frameon=False)
query_figure_path = figure_directory / "01_conditional_queries.png"
query_figure.savefig(query_figure_path, dpi=200, bbox_inches="tight")
for title, error in query_errors.items():
    print(f"{title} RMSE: {error:.4f}")
print(f"Figure saved to: {query_figure_path}")
query_figure

Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/00_rank3_close_spectrum.png
Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/00_rank3_training_loss.png


$\mathbb{E}[Y\mid X=x]$ RMSE: 0.0319
$\mathrm{Var}(Y\mid X=x)$ RMSE: 0.0192
$\mathbb{P}(Y>0.5\mid X=x)$ RMSE: 0.0234
Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/01_conditional_queries.png


<Figure size 1200x320 with 3 Axes>

## Random-forest weak learners

We repeat the kernel estimation and conditional queries with random forests in place of individual regression trees.

In [6]:
forest_parameters = {
    "n_iterations": 10,
    "step_size": 0.1,
    "max_depth": 3,
    "min_samples_leaf": 300,
    "learner_type": "random_forest",
    "n_estimators": 30,
}
phi_forest, psi_forest, values_forest, history_forest = fit_fsnm(
    x_train, y_train, rank=RANK, seed=SEED, **forest_parameters
)
phi_forest_validation, psi_forest_validation = scaled_factors(
    phi_forest, psi_forest, values_forest, x_validation, y_validation
)
forest_validation_loss = float(
    empirical_loss(phi_forest_validation, psi_forest_validation)
)
phi_forest_grid = phi_forest.predict(grid[:, None])
psi_forest_grid = psi_forest.predict(grid[:, None])
kappa_forest = 1 + (phi_forest_grid * values_forest) @ psi_forest_grid.T
forest_metrics = {
    "RMSE": np.sqrt(np.mean((kappa_forest - kappa_true) ** 2)),
    "spectrum error": np.linalg.norm(values_forest - SIGMAS),
    "subspace error": 0.5 * (
        subspace_error(phi_forest_grid, exact_basis)
        + subspace_error(psi_forest_grid, exact_basis)
    ),
    "orthogonality error": 0.5 * (
        orthogonality_error(phi_forest_grid)
        + orthogonality_error(psi_forest_grid)
    ),
}
print(f"Random-forest parameters: {forest_parameters}")
print(f"Estimated singular values: {np.round(values_forest, 4)}")
print(f"Validation loss: {forest_validation_loss:.4f}")
for metric, value in forest_metrics.items():
    print(f"{metric:<24}{value:.4f}")

forest_figure, forest_axes = plt.subplots(
    1, 3, figsize=(11.5, 3.6), constrained_layout=True
)
for axis, (values, title) in zip(
    forest_axes[:2],
    [(kappa_true, r"True $\kappa$"), (kappa_forest, r"FSNM $\widehat\kappa$")],
):
    image = axis.imshow(
        values.T, origin="lower", extent=[-1, 1, -1, 1], cmap="coolwarm",
        vmin=value_limits[0], vmax=value_limits[1],
    )
    axis.set(title=title, xlabel="$x$", ylabel="$y$")
    forest_figure.colorbar(image, ax=axis, shrink=0.82)
forest_error = kappa_forest - kappa_true
forest_error_limit = np.max(np.abs(forest_error))
image = forest_axes[2].imshow(
    forest_error.T, origin="lower", extent=[-1, 1, -1, 1], cmap="coolwarm",
    vmin=-forest_error_limit, vmax=forest_error_limit,
)
forest_axes[2].set(title=r"$\widehat\kappa-\kappa$", xlabel="$x$", ylabel="$y$")
forest_figure.colorbar(image, ax=forest_axes[2], shrink=0.82)
forest_figure_path = figure_directory / "02_forest_rank3.png"
forest_figure.savefig(forest_figure_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {forest_figure_path}")

forest_iterations = np.arange(1, len(history_forest["training_loss"]) + 1)
forest_loss_figure, forest_loss_axis = plt.subplots(
    figsize=(5.5, 3.6), constrained_layout=True
)
forest_loss_axis.plot(
    forest_iterations, history_forest["training_loss"],
    color="tab:blue", linewidth=2,
)
forest_loss_axis.set(xlabel="Iteration", ylabel="Empirical training loss")
forest_loss_axis.grid(alpha=0.25)
forest_loss_path = figure_directory / "02_forest_training_loss.png"
forest_loss_figure.savefig(forest_loss_path, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {forest_loss_path}")

psi_forest_reference = psi_forest.predict(y_reference[:, None])
kappa_forest_queries = (
    1 + (phi_forest_grid * values_forest) @ psi_forest_reference.T
)
def forest_conditional(query_values):
    return np.mean(kappa_forest_queries * query_values[None, :], axis=1)

forest_mean = forest_conditional(y_reference)
forest_second_moment = forest_conditional(y_reference**2)
forest_variance = forest_second_moment - forest_mean**2
forest_tail = forest_conditional((y_reference > 0.5).astype(float))
forest_query_results = [
    (r"$\mathbb{E}[Y\mid X=x]$", exact_mean, forest_mean),
    (r"$\mathrm{Var}(Y\mid X=x)$", exact_variance, forest_variance),
    (r"$\mathbb{P}(Y>0.5\mid X=x)$", exact_tail, forest_tail),
]
forest_query_figure, forest_query_axes = plt.subplots(
    1, 3, figsize=(12, 3.2), constrained_layout=True
)
for axis, (title, exact, estimated) in zip(
    forest_query_axes, forest_query_results
):
    axis.plot(grid, exact, color="black", linewidth=2, label="Exact")
    axis.plot(
        grid, estimated, color="tab:red", linewidth=2,
        linestyle="--", label="FSNM",
    )
    axis.set(title=title, xlabel="$x$")
forest_query_axes[0].set_ylabel("Conditional functional")
forest_query_axes[0].legend(frameon=False)
forest_query_path = figure_directory / "03_forest_conditional_queries.png"
forest_query_figure.savefig(forest_query_path, dpi=200, bbox_inches="tight")
for title, exact, estimated in forest_query_results:
    error = np.sqrt(np.mean((estimated - exact) ** 2))
    print(f"{title} RMSE: {error:.4f}")
print(f"Figure saved to: {forest_query_path}")
forest_query_figure

Random-forest parameters: {'n_iterations': 10, 'step_size': 0.1, 'max_depth': 3, 'min_samples_leaf': 300, 'learner_type': 'random_forest', 'n_estimators': 30}
Estimated singular values: [0.1799 0.1333 0.1005]
Validation loss: -0.0512
RMSE                    0.1082
spectrum error          0.0331
subspace error          0.3061
orthogonality error     0.0232


Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/02_forest_rank3.png
Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/02_forest_training_loss.png


$\mathbb{E}[Y\mid X=x]$ RMSE: 0.0298
$\mathrm{Var}(Y\mid X=x)$ RMSE: 0.0130
$\mathbb{P}(Y>0.5\mid X=x)$ RMSE: 0.0224
Figure saved to: /home/thiago/Projects/paper-fsnm/code/fsnm/figures/03_forest_conditional_queries.png


<Figure size 1200x320 with 3 Axes>